# Notebook 05 — Decision-Flip Instability and Digital Perturbation Robustness Analysis

**Purpose:** Audit the pre-acquisition decision-flip head across $M = 14,815$ realized on-policy test transitions (`ROC AUROC = 0.9473`, `AP = 0.5172`, `Brier = 0.0416`), distinguish threshold-free ranking from operational alert deployment, and analyze controlled digital perturbation sensitivity across `16` transformations on held-out Validation `A0` (`N = 50` images).  
**Data & Artifact Requirement:** **Standalone Clean-Room Capable.** Executes completely from tracked canonical audit JSONs and benchmark tables (`decision_flip_cluster_results.json`, `P5_6_TRANSITION_MATRICES.json`, `tab5_decision_flip_audit.tex`, `tab7_perturbation_robustness.tex`).

## 1. Learning Objectives & Scientific Context

### Learning Objectives
1. Inspect pre-acquisition decision-flip ranking performance (`AUROC`, `Step-Function AP`, `Brier`) and equal-width calibration bins over realized on-policy transitions.
2. Explain clearly why high threshold-free ranking discrimination (`AUROC = 0.9473` at `6.67%` prevalence) **does not** establish a clinically validated runtime alert or error-prevention system.
3. Compare single-view (`A0`) digital perturbation sensitivity across spatial/geometric crops, optical/compression artifacts, and photometric adjustments (`N = 50`, `800` observations) under Estimands A, B, and C.

### Prerequisites
- Core packages (`numpy`, `pandas`, `matplotlib`).

## 2. Environment & Canonical Artifact Preflight

In [ ]:
from __future__ import annotations

import json
import os
import platform
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt


def find_repo_root(start: Path | None = None) -> Path:
    """Locate repository root portably without hardcoded machine paths."""
    cur = (start or Path.cwd()).resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "research" / "derma_act" / "config.py").is_file():
            return candidate
    raise FileNotFoundError("Could not locate DERMA-ACT repository root containing pyproject.toml and research/derma_act/config.py")


def find_tables_dir(repo_root: Path) -> Path | None:
    for rel in [
        Path("supplementary") / "tables",
        Path("manuscripts") / "current" / "PAPER_DRAFT_Q2_V3_2_1_PRE_SUBMISSION" / "tables",
    ]:
        cand = repo_root / rel
        if cand.is_dir():
            return cand
    return None


SEED = 42
random.seed(SEED)
np.random.seed(SEED)

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

OUT_DIR = REPO_ROOT / "outputs" / "notebooks" / "nb05_flip_robustness"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FLIP_JSON = REPO_ROOT / "research" / "derma_act" / "p5_5_final_claim_closure" / "decision_flip_cluster_results.json"
TRANS_JSON = REPO_ROOT / "research" / "derma_act" / "p5_6_action_quality_audit" / "P5_6_TRANSITION_MATRICES.json"
TABLES_DIR = find_tables_dir(REPO_ROOT)
TAB5_TEX = TABLES_DIR / "tab5_decision_flip_audit.tex" if TABLES_DIR else Path("__missing__")
TAB7_TEX = TABLES_DIR / "tab7_perturbation_robustness.tex" if TABLES_DIR else Path("__missing__")

display(pd.DataFrame([
    {"artifact": "decision_flip_cluster_results.json", "available": FLIP_JSON.is_file(), "tier": "TRACKED"},
    {"artifact": "P5_6_TRANSITION_MATRICES.json", "available": TRANS_JSON.is_file(), "tier": "TRACKED"},
    {"artifact": "tab5_decision_flip_audit.tex", "available": TAB5_TEX.is_file(), "tier": "TRACKED"},
    {"artifact": "tab7_perturbation_robustness.tex", "available": TAB7_TEX.is_file(), "tier": "TRACKED"},
]))

## 3. Methodology: Instability Head Scope vs. Operational Alerts

At step $t$, `MultiHeadActionPolicy` outputs a scalar logit $\mathrm{logit}_i(s_t, a)$ whose sigmoid $\sigma(\mathrm{logit}_i)$ estimates the probability that acquiring action $a$ changes the Top-1 `argmax` class on the next step: $I_t = \mathbb{I}[\hat{y}_{t+1} \neq \hat{y}_t]$.

- **What this measures:** Pre-acquisition ranking over $M = 14,815$ realized on-policy test transitions ($988$ flips = `6.67%` prevalence across $K = 1,887$ multi-view groups).
- **What this does NOT measure:** No runtime intervention threshold was prespecified, calibrated prospectively, or evaluated for clinical error prevention.

Separately, perturbation robustness evaluates single global-view (`A0`) prediction stability on an isolated held-out Validation subset ($N = 50$ images: `47` unique lesion IDs + `3` singletons; `0` overlap with Test) across `16` digital transformations using Total Variation Distance (TVD), Top-1 consistency, and entropy shift $|\Delta H|$.

In [ ]:
import re

flip_res = json.loads(FLIP_JSON.read_text(encoding="utf-8"))
flip_summary = pd.DataFrame([
    {"metric": "Realized on-policy transitions (M)", "estimate": f"{flip_res['sample_size_transitions']:,}", "ci95": f"K={flip_res['unique_lesion_clusters']:,} clusters"},
    {"metric": "Observed Top-1 flips (prevalence)", "estimate": f"{flip_res['observed_flips']} ({100*flip_res['prevalence']:.2f}%)", "ci95": f"[{100*flip_res['prevalence_ci95'][0]:.2f}%, {100*flip_res['prevalence_ci95'][1]:.2f}%]"},
    {"metric": "ROC AUROC (threshold-free ranking)", "estimate": f"{flip_res['auroc']:.4f}", "ci95": f"[{flip_res['auroc_ci95'][0]:.4f}, {flip_res['auroc_ci95'][1]:.4f}]"},
    {"metric": "Average Precision (step-function AP)", "estimate": f"{flip_res['auprc']:.4f}", "ci95": f"[{flip_res['auprc_ci95'][0]:.4f}, {flip_res['auprc_ci95'][1]:.4f}]"},
    {"metric": "Brier Score", "estimate": f"{flip_res['brier_score']:.4f}", "ci95": f"[{flip_res['brier_ci95'][0]:.4f}, {flip_res['brier_ci95'][1]:.4f}]"},
])
display(flip_summary)

# Parse Panel B instability bins from canonical tab5_decision_flip_audit.tex
bin_rows = []
for ln in TAB5_TEX.read_text(encoding="utf-8").splitlines():
    m = re.match(r"^\{\[\}([0-9.]+),\s*([0-9.]+)[\)\]].*&", ln.strip())
    if m:
        parts = [p.strip() for p in ln.replace("\\\\", "").split("&")]
        n_trans = int(parts[1].replace(",", ""))
        if n_trans == 0:
            continue
        flips = int(parts[2].replace(",", ""))
        m_rate = re.search(r"([0-9.]+)\s*\$\\to\$\s*([0-9.]+)\s*\[([0-9.]+),\s*([0-9.]+)\]", parts[4])
        bin_rows.append({
            "bin": f"[{m.group(1)}, {m.group(2)})",
            "n_transitions": n_trans,
            "realized_flips": flips,
            "mean_pred_score": float(m_rate.group(1)),
            "obs_flip_rate": float(m_rate.group(2)),
            "ci_low": float(m_rate.group(3)),
            "ci_high": float(m_rate.group(4)),
        })
bins_df = pd.DataFrame(bin_rows)
display(bins_df)

# Parse canonical 16 perturbation conditions from tab7_perturbation_robustness.tex
pert_rows = []
for ln in TAB7_TEX.read_text(encoding="utf-8").splitlines():
    if "&" in ln and ("Spatial / Geometric" in ln or "Optical / Compression" in ln or "Photometric" in ln or "Identity Control" in ln):
        parts = [p.strip() for p in ln.replace("\\\\", "").split("&")]
        m_tvd = re.search(r"([0-9.]+)\s*\[([0-9.]+),\s*([0-9.]+)\]", parts[3])
        pert_rows.append({
            "condition": re.sub(r"\$[^$]*\$", lambda s: s.group(0).replace("$", "").replace("\\sigma", "sigma").replace("\\gamma", "gamma"), parts[0]),
            "family": parts[1],
            "consistency_pct": float(parts[2].replace("\\%", "")),
            "tvd_a": float(m_tvd.group(1)),
            "tvd_a_low": float(m_tvd.group(2)),
            "tvd_a_high": float(m_tvd.group(3)),
            "tvd_b": float(parts[4]),
            "tvd_c": float(parts[5]),
            "mean_abs_dH": float(parts[6]),
        })
pert_df = pd.DataFrame(pert_rows)
display(pert_df)

## 4. Visualization: Instability Reliability Curve & Digital Perturbation Sensitivity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))

# Panel A: Instability head calibration / reliability curve across realized on-policy transitions
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.6, label="Ideal Calibration (y = x)")
axes[0].errorbar(
    bins_df["mean_pred_score"],
    bins_df["obs_flip_rate"],
    yerr=[bins_df["obs_flip_rate"] - bins_df["ci_low"], bins_df["ci_high"] - bins_df["obs_flip_rate"]],
    fmt="o-", color="#2b6cb0", capsize=3, linewidth=2, label="Observed Flip Rate [95% CI]",
)
for _, r in bins_df.iterrows():
    if r["n_transitions"] <= 30:
        axes[0].annotate(f"sparse n={r['n_transitions']}", (r["mean_pred_score"], r["obs_flip_rate"]), xytext=(5, -10), textcoords="offset points", fontsize=7.5, color="#c53030")
axes[0].set_xlim(-0.02, 0.92); axes[0].set_ylim(-0.02, 1.05)
axes[0].set_xlabel("Mean Predicted Instability Score sigma(logit_i)")
axes[0].set_ylabel("Observed Next-Step Top-1 Flip Rate")
axes[0].set_title(f"Decision-Flip Head Reliability (M={flip_res['sample_size_transitions']:,}, AUROC={flip_res['auroc']:.4f})")
axes[0].legend(fontsize=8, loc="upper left")
axes[0].grid(alpha=0.3)

# Panel B: Perturbation robustness ranked by Estimand A Mean TVD
p_non_id = pert_df[pert_df["family"] != "Identity Control"].sort_values("tvd_a", ascending=True)
fam_colors = {"Spatial / Geometric": "#c53030", "Optical / Compression": "#dd6b20", "Photometric": "#2b6cb0"}
colors = [fam_colors.get(f, "#718096") for f in p_non_id["family"]]
axes[1].barh(p_non_id["condition"], p_non_id["tvd_a"], color=colors,
             xerr=[p_non_id["tvd_a"] - p_non_id["tvd_a_low"], p_non_id["tvd_a_high"] - p_non_id["tvd_a"]], capsize=2)
axes[1].set_xlabel("Mean Total Variation Distance (Estimand A, 95% CI)")
axes[1].set_title("A0 Perturbation Sensitivity (Val N=50; Red=Spatial, Orange=Optical, Blue=Photo)")
axes[1].grid(axis="x", alpha=0.3)

plt.tight_layout()
fig.savefig(OUT_DIR / "flip_and_perturbation_analysis.png", dpi=150)
plt.show()

## 5. Interpretation & Scientific Boundaries

- **Ranking vs. Clinical Alerting:** While `ROC AUROC = 0.9473` and `AP = 0.5172` (`7.76x` baseline flip prevalence `6.67%`) show strong discrimination over realized on-policy transitions, the head predicts whether the next algorithmic view changes `argmax` — it does **not** detect clinical errors directly, and **no runtime alert threshold has been validated**.
- **Sparse Upper Bins:** Bins `[0.7, 0.8)` (`n = 30`) and `[0.8, 0.9)` (`n = 6`) are small, and `[0.9, 1.0]` is empty (`n = 0`).
- **Perturbation Scope:** Spatial framing perturbations (`5%` center crop TVD `0.0862`; ±5 px translation TVD `0.0784–0.0813`) shift `A0` posteriors 3–6x more than ±5% photometric/gamma shifts (`0.0127–0.0233`), consistent with spatial sensitivity on `N = 50` validation images; however, controlled digital transforms do not prove real-world optical or multi-center camera invariance.

### Reproduction & References
- Output saved to: `outputs/notebooks/nb05_flip_robustness/flip_and_perturbation_analysis.png`
- Related scripts/docs: [`scripts/figure_sources/fig6_decision_flip_prediction.py`](../scripts/figure_sources/fig6_decision_flip_prediction.py) · [`scripts/figure_sources/fig8_perturbation_robustness.py`](../scripts/figure_sources/fig8_perturbation_robustness.py) · [Limitations](../docs/LIMITATIONS.md)